In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from google.colab import files


In [ ]:
uploaded=files.upload()

df=pd.read_csv("heart.csv")
df.head()

Saving heart.csv to heart.csv


,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up,0
1,49,F,NAP,160,180,0,Normal,156,N,1.0,Flat,1
2,37,M,ATA,130,283,0,ST,98,N,0.0,Up,0
3,48,F,ASY,138,214,0,Normal,108,Y,1.5,Flat,1
4,54,M,NAP,150,195,0,Normal,122,N,0.0,Up,0


# Polynomial Features create

In [ ]:
# Polynomial features using Age and MaxHR
from sklearn.preprocessing import PolynomialFeatures

poly_cols=["Age", "MaxHR"]
ploy=PolynomialFeatures(degree=2, include_bias=False)

poly_features= ploy.fit_transform(df[poly_cols])
poly_features_names= ploy.get_feature_names_out(poly_cols)

print(poly_features_names)
poly_features.shape

['Age' 'MaxHR' 'Age^2' 'Age MaxHR' 'MaxHR^2']


(918, 5)

In [ ]:
# Binning Age into categoriew (Young, Middle, Old) আমরা যে custom encodng করতাম, অনেকটা তেমনই

df["Age_bin"]=pd.cut(
    df["Age"],
    bins=[0,30,50,100],  # bins হল বাকেটের মত, আমরা যে কয়টা range বানাইতে চাই, তার থেকে ১ টা বেশি নিব
    labels=["Young", "Middle", "Old"]
)

df[["Age", "Age_bin"]].head(10)

,Age,Age_bin
0,40,Middle
1,49,Middle
2,37,Middle
3,48,Middle
4,54,Old
5,39,Middle
6,45,Middle
7,54,Old
8,37,Middle
9,48,Middle


In [ ]:
# Domain -driven risk categories for RestingBP and OldPeak

def bp_risk(bp):
  if bp<120:
    return "Normal"
  elif bp<140:
    return "Elevated"
  else:
    return "High"

def oldpeak_risk(op):
  if op==0:
    return "No Stress"
  elif op<2:
    return "Moderate Stress"
  else:
    return "High Stress"

df["BP_Risk"]=df["RestingBP"].apply(bp_risk)
df["Oldpeak_Risk"]=df["Oldpeak"].apply(oldpeak_risk)


df[["RestingBP", "BP_Risk", "Oldpeak", "Oldpeak_Risk"]].head(10)

,RestingBP,BP_Risk,Oldpeak,Oldpeak_Risk
0,140,High,0.0,No Stress
1,160,High,1.0,Moderate Stress
2,130,Elevated,0.0,No Stress
3,138,Elevated,1.5,Moderate Stress
4,150,High,0.0,No Stress
5,120,Elevated,0.0,No Stress
6,130,Elevated,0.0,No Stress
7,110,Normal,0.0,No Stress
8,140,High,1.5,Moderate Stress
9,120,Elevated,0.0,No Stress


# Pipelline

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder

In [ ]:
# Define numeric and categorical columns
num_features=["Age", "RestingBP", "Cholesterol", "MaxHR", "Oldpeak"]
cat_features=["Sex", "ChestPainType", "RestingECG", "ExerciseAngina", "ST_Slope"]

In [ ]:
# Numeric pipline
num_pippeline=Pipeline([
    ("scaler", StandardScaler())
])

# Categorical pipline
cat_pipline=Pipeline([
    ("ohe", OneHotEncoder(drop="first"))
])

# Combine both
preprocess= ColumnTransformer([
    ("num", num_pippeline, num_features),
    ("cat", cat_pipline, cat_features)
])

# full pipeline with a simple model
clf= Pipeline([
    ("prep", preprocess),
    ("model", LogisticRegression(max_iter=1000))
])

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
# train-test split
X= df.drop(columns=["HeartDisease"])
y= df["HeartDisease"]

X_train_pipe, X_test_pipe, y_train_pipe, y_test_pipe= train_test_split(X, y, test_size=0.25, random_state= 42)

# fit the full pipeline
clf.fit(X_train_pipe, y_train_pipe)

# predict and evaluate
y_pred_pipe= clf.predict(X_test_pipe)
acc= accuracy_score(y_test_pipe, y_pred_pipe)
print(acc)

0.8434782608695652
